**CI twin of `ch10-decision-trees.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from collections import Counter

df = load_csv("penguins").dropna(subset=["bill_length_mm",
                                         "flipper_length_mm"])
y = df["species"]

def gini(labels):
    n = len(labels)
    return 1 - sum((c / n) ** 2 for c in Counter(labels).values())

print(f"whole colony (3 species mixed): gini = {gini(y.tolist()):.3f}\n")

candidates = [("bill_length_mm", 43.0),
              ("bill_depth_mm", 17.0),
              ("flipper_length_mm", 207.0)]
for col, thr in candidates:
    left = y[df[col] <= thr].tolist()
    right = y[df[col] > thr].tolist()
    score = (len(left) * gini(left) + len(right) * gini(right)) / len(y)
    print(f"{col} <= {thr}:")
    print(f"   left  {len(left):3} birds, gini {gini(left):.3f}   "
          f"right {len(right):3} birds, gini {gini(right):.3f}   "
          f"-> weighted {score:.3f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

feats = ["flipper_length_mm", "bill_length_mm"]
Xtr, Xte, ytr, yte = train_test_split(
    df[feats], y, test_size=0.25, random_state=42, stratify=y)

tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xtr, ytr)
print(export_text(tree, feature_names=feats))
print(f"held-out accuracy: {accuracy_score(yte, tree.predict(Xte)):.3f}")

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler().fit(Xtr)
scaled_tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(
    sc.transform(Xtr), ytr)
print(f"raw features:    {accuracy_score(yte, tree.predict(Xte)):.3f}")
print(f"scaled features: "
      f"{accuracy_score(yte, scaled_tree.predict(sc.transform(Xte))):.3f}")

In [ ]:
print("depth      train acc   test acc")
for depth in (1, 2, 3, 5, None):
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(Xtr, ytr)
    tr_acc = accuracy_score(ytr, t.predict(Xtr))
    te_acc = accuracy_score(yte, t.predict(Xte))
    print(f"{str(depth):5}       {tr_acc:.3f}      {te_acc:.3f}")

full = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
print(f"\nunlimited tree: depth {full.get_depth()}, "
      f"{full.get_n_leaves()} leaves for 256 training birds")

In [ ]:
for name, imp in zip(feats, tree.feature_importances_):
    print(f"{name:20} {imp:.3f}")

In [ ]:
print(f"full training set root: flipper_length_mm <= 206.50")
for seed in (0, 1):
    s = df.sample(n=100, random_state=seed)
    t = DecisionTreeClassifier(max_depth=2, random_state=0).fit(
        s[feats], s["species"])
    root = feats[t.tree_.feature[0]]
    print(f"100-bird sample {seed} root:   {root} <= "
          f"{t.tree_.threshold[0]:.2f}")

In [ ]:
model = DecisionTreeClassifier(max_depth=2, random_state=0)
model.fit(Xtr, ytr)

run_tests([
    ("held-out accuracy", round(
        accuracy_score(yte, model.predict(Xte)), 3), 0.977),
    ("the root question is about flippers",
     feats[model.tree_.feature[0]], "flipper_length_mm"),
    ("at the learned threshold", round(
        float(model.tree_.threshold[0]), 2), 206.5),
])

In [ ]:
from collections import Counter

def gini(labels):
    n = len(labels)
    return 1 - sum((c / n) ** 2 for c in Counter(labels).values())

def split_score(left_labels, right_labels):
    n = len(left_labels) + len(right_labels)
    return (len(left_labels) * gini(left_labels)
            + len(right_labels) * gini(right_labels)) / n

run_tests([
    ("a pure group", gini(["A", "A", "A", "A"]), 0.0),
    ("a 50/50 mix", gini(["A", "B", "A", "B"]), 0.5),
    ("two thirds / one third", round(gini(["A", "A", "B"]), 4), 0.4444),
    ("a perfect split scores 0",
     split_score(["A", "A", "A"], ["B", "B"]), 0.0),
    ("the chapter's fixture", round(
     split_score(["A"] * 5, ["B", "B", "B", "A", "A"]), 4), 0.24),
], tol=1e-9)